# Optunaによるパラメータのオートチューニング

In [2]:
!pip install -qq optuna kaggle-environments

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.7/721.7 kB 14.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 944.3/944.3 kB 52.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.3/96.3 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 53.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 840.2/840.2 kB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.4/111.4 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.4/267.4 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.2/20.2 MB 44.1 MB

In [3]:
"""Colabで実行するパラメータ探索コード。"""

import importlib
import statistics

import optuna
from kaggle_environments import make

import main as strategy


# Colab上の最新のmain.pyを読み込む。
strategy = importlib.reload(strategy)

N_TRIALS = 150
MATCH_COUNT = 20
EVALUATION_SEEDS = list(range(MATCH_COUNT))


def objective(trial):
    """平均得点を返す。"""

    #PET_CAFE1店舗あたりのニンジン目標数
    strategy.StrategyConfig.CARROT_TARGET_PER_PET_CAFE = trial.suggest_int(
        "CARROT_TARGET_PER_PET_CAFE",
        3,
        12,
    )

    #作業対象探索から雑草と空き地を除外し始める日
    strategy.StrategyConfig.GENERAL_PLANT_END_DAY = trial.suggest_int(
        "GENERAL_PLANT_END_DAY",
        22,
        29,
    )

    #小麦種がない場合に購入する数量
    strategy.StrategyConfig.WHEAT_SEED_BUY_COUNT = trial.suggest_int(
        "WHEAT_SEED_BUY_COUNT",
        2,
        10,
    )

    #牛の餌として最低限確保する小麦数
    strategy.StrategyConfig.MIN_FEED_WHEAT = trial.suggest_int(
        "MIN_FEED_WHEAT",
        1,
        6,
    )


    rewards = []

    for episode_seed in EVALUATION_SEEDS:
        strategy.hire_controller = strategy.HireController()

        env = make(
            "kaggriculture",
            configuration={
                "episodeSteps": 720,
                "seed": episode_seed,
            },
            debug=True,
        )

        env.run([strategy.agent, strategy.agent])

        for state in env.steps[-1]:
            rewards.append(float(state.reward))

    return statistics.fmean(rewards)


study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
)

study.optimize(
    objective,
    n_trials=N_TRIALS,
    n_jobs=1,
    show_progress_bar=True,
)

print("\n===== 最高平均得点 =====")
print(study.best_value)

print("\n===== main.pyへ手動設定する値 =====")

for name, value in study.best_params.items():
    print(f"{name} = {value}")

[I 2026-09-09 23:40:54,633] A new study created in memory with name: no-name-ce0c7aac-f19e-49b8-968a-1e165e326013


  0%|          | 0/150 [00:00<?, ?it/s]

[I 2026-09-09 23:44:04,418] Trial 0 finished with value: 61813.5 and parameters: {'CARROT_TARGET_PER_PET_CAFE': 6, 'GENERAL_PLANT_END_DAY': 29, 'WHEAT_SEED_BUY_COUNT': 8, 'MIN_FEED_WHEAT': 4}. Best is trial 0 with value: 61813.5.
[I 2026-09-09 23:47:13,103] Trial 1 finished with value: 59129.55 and parameters: {'CARROT_TARGET_PER_PET_CAFE': 4, 'GENERAL_PLANT_END_DAY': 23, 'WHEAT_SEED_BUY_COUNT': 2, 'MIN_FEED_WHEAT': 6}. Best is trial 0 with value: 61813.5.
[I 2026-09-09 23:50:19,992] Trial 2 finished with value: 58477.625 and parameters: {'CARROT_TARGET_PER_PET_CAFE': 9, 'GENERAL_PLANT_END_DAY': 27, 'WHEAT_SEED_BUY_COUNT': 2, 'MIN_FEED_WHEAT': 6}. Best is trial 0 with value: 61813.5.
[I 2026-09-09 23:53:25,790] Trial 3 finished with value: 63090.55 and parameters: {'CARROT_TARGET_PER_PET_CAFE': 11, 'GENERAL_PLANT_END_DAY': 23, 'WHEAT_SEED_BUY_COUNT': 3, 'MIN_FEED_WHEAT': 2}. Best is trial 3 with value: 63090.55.
[I 2026-09-09 23:56:36,443] Trial 4 finished with value: 65814.575 and par